# Frequent Itemset Mining
**Prepared by Christian Alis**

Frequent Itemset Mining is also known as Frequent Pattern Mining. In practice, we don't usually end with frequent itemset mining but instead continues on to association pattern mining.

Frequent Itemset Mining and Association Pattern Mining (aka Association Rule Mining) were born out of market basket analysis. Hence, many of the terms used are relevant to that original problem.

Consider a _database_ $\mathscr{T}$, which is an unordered set of $N$ transactions $\{T_1, T_2, ..., T_N\}$. Each transaction $T_i$ is a set of $n_i = \left|T_i\right|$ items drawn from a universe $\mathscr{U}$ of items. Instead of using a sequence of sets, we may represent $\mathscr{T}$ as a matrix of $d=\left|\mathscr{U}\right|$ dimensions where 0 represents the absence and, a positive number, the presence of the item in the transaction.

In this notebook, we will only consider the presence of an item in a transaction and will ignore the actual amount (as long as it's nonzero, of course). Furthermore, in the real world, $d$ is large.

The following is an example of a database:

| tid |           items             |
|-----|-----------------------------|
|  1  | {bread, butter, milk}       |
|  2  | {eggs, milk, yogurt}        |
|  3  | {bread, cheese, eggs, milk} |
|  4  | {eggs, milk, yogurt}        |
|  5  | {cheese, milk, yogurt}      |

In Python, we may represent the database as a list of sets, making use of the fact that the `tid` is sequential.

**Problem 1**

Give three examples of business problems that can be solved using Frequent Itemset Mining or Associative Pattern Mining.

Example 1. **Market Basket Analysis**: Retailers can analyze customer purchase data to identify frequently bought together items. For example, if customers often buy bread and butter together, the store can place these items near each other or offer discounts on bundled purchases.

Example 2. **Cross-Selling Opportunities**: E-commerce platforms can use frequent itemset mining to recommend additional products to customers based on their current shopping cart. For instance, if a customer adds a laptop to their cart, the system can suggest accessories like a mouse or laptop bag that are frequently purchased together with laptops.

Example 3: **Healthcare Treatment Patterns**: Hospitals and healthcare providers can analyze patient treatment data to identify common combinations of medications or therapies that are frequently prescribed together. This can help in optimizing treatment plans, improving patient outcomes, and reducing costs by identifying effective treatment combinations.

In [1]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from numpy.testing import assert_equal, assert_array_equal
from itertools import combinations

In [2]:
db = [{"bread", "butter", "milk"},
      {"eggs", "milk", "yogurt"},
      {"bread", "cheese", "eggs", "milk"},
      {"eggs", "milk", "yogurt"},
      {"cheese", "milk", "yogurt"}]

We may also represent the database as a sparse matrix.

**Problem 2**

Create a function `to_sparse` that reads `db` and returns a numpy sparse matrix representation of the database. Sort the columns in alphabetical order.

In [3]:
def to_sparse(db):
    universe = sorted(set().union(*db))

    table = []

    for transaction in db:
        row = []
        for item in universe:
            if item in transaction:
                row.append(1)
            else:
                row.append(0)
        table.append(row)

    result = csr_matrix(table)

    return result

In [4]:
db_sparse = to_sparse(db)
assert isinstance(db_sparse, csr_matrix)
assert_equal(db_sparse.shape, (5, 6))
assert_array_equal(
    db_sparse[0, :].toarray().squeeze(), [1.0, 1.0, 0.0, 0.0, 1.0, 0.0]
)

An **itemset** $X$ is a subset of items, $X \subseteq \mathscr{U}$. A length $k = |X|$ itemset is also known as a **$k$-itemset**. For example, the subset itemsets of {bread, butter, milk} are {bread}, {butter}, {milk}, {bread, butter}, {bread, milk}, {butter, milk}, {bread, butter, milk}. It consists of three 1-itemsets, three 2-itemsets and one 3-itemset.

For our course, we define the **support** $sup(X)$ of an itemset $X$ as the number of transactions having $X$ as a subset,
$$
sup(X) = \left|{T_i|X \subseteq T_i}\right|.
$$
This is known as the **absolute support**. We define **relative support** $relSup(X)$ as the fraction of transactions having $X$ as a subset,
$$
relSup(X) = \frac{sup(X)}{N}.
$$
For many references and libraries, support is defined as the _relative_ support so be careful when reading or using other materials. It's quite easy to convert between absolute support and relative support, though. To illustrate, the (absolute) support for {bread, milk} is 2 and its relative support is $2/5=0.4$.

We define **frequent itemsets** (or frequent patterns) as those itemsets that have a support at least equal to a given minimum support ($minsup$). The task of frequent itemset mining is to find all frequent itemsets in the database.

In this notebook, we look at three methods for finding the frequent itemsets:
* Brute force
* Apriori
* FP-growth

## Brute force algorithm

The brute force approach to frequent itemset mining is to compute the support of each possible itemset of $\mathscr{U}$ then return only those itemsets whose support pass the threshold $minsup$.

**Problem 3**

Create a function `brute` that accepts `db` and `minsup` then returns the frequent itemsets as a list of tuples with their support using the brute force algorithm. Sort the items in each tuple alphabetically. Sort the results, first by decreasing support, then by decreasing number of items then alphabetically. Use only the Python Standard Library.

In [5]:
def brute(db, minsup):
    universe = sorted(set().union(*db))

    frequent = []

    for size in range(1, len(universe) + 1):
        for combo in combinations(universe, size):
            support = sum(1 for item in db if set(combo).issubset(item))
            if support >= minsup:
                frequent.append((combo, support))

    frequent.sort(key=lambda x: (-x[1], -len(x[0]), x[0]))

    return frequent

In [6]:
fi_brute = brute(db, 2)
assert_equal(len(fi_brute), 11)
assert_equal(fi_brute[0], (('milk',), 5))

Although it took you a few loops, the brute force algorithm is relatively simple. It doesn't scale well though.

**Problem 4**

Estimate the runtime for `brute`, relative to the runtime for the current case, when the number of items is 1000 but the number of transactions remains the same, and when the number of transactions is 1000 but the number of items remains the same.


## Apriori algorithm

Clearly, generating all the possible itemsets from a universe of items could take a while. Coupled with scanning through the entire dataset for each possible itemset, the brute force algorithm is simply untenable for large databases. To make progress in making frequent itemset mining scalable, one should realize that generating the itemsets can be thought of as doing a search. Figure 1 shows the search space for a 5-item database.

<div style="width: 40em; margin: 0 auto">
    <img src="search-space.png" style="width: 25em" />
    <p><strong>Figure 1</strong>. The $k$-th layer corresponds to the $k$-itemsets for the 5-item database. A line connects a $k$-itemset to a $(k+1)$-itemset generated by adding a unique item to the latter. The tree can be partitioned into frequent and infrequent itemsets using a border (blue broken line). Image source: Aggarwal (2015).</p>
</div>

We now introduce the following important properties of itemsets:

* **Support monoticity property**: The support of every subset $Y$ of itemset $X$ is at least equal to that of the support for $X$, $sup(Y) \geq sup(X), \forall Y \subseteq X$.
* **Downward closure property**: Every subset of a frequent itemset is also frequent.
* **Maximal frequent itemsets**: A frequent itemset is maximal at a given minimum support $minsup$ if it is frequent and no superset of it is frequent.

Using the properties above, what we can do is to perform a breadth-first search on the tree starting from the 1-itemsets, we check whether the itemset is frequent and we only go deeper if it is. This is the gist of the Apriori algorithm which we formally write down below.

<img src="apriori.png" style="width: 40em" />

An implication of the Apriori algorithm is that we would be able partition the search space by a border (Figure 1). The frequent itemsets along the border are the maximal frequent itemsets.

**Problem 5**

Create a function `apriori` that accepts `db` and `minsup` then returns the frequent itemsets as a list of tuples with their support using the Apriori algorithm. Sort the items in each tuple alphabetically. Sort the results, first by decreasing support, then by decreasing number of items then alphabetically. Use only the Python Standard Library.

In [7]:
def apriori(db, minsup):
    universe = sorted(set().union(*db))
    k = 1
    frequent = []
    L_prev = None

    while True:
        L_k = []
        if k == 1:
            C_k = [(item,) for item in universe]
        else:
            candidates = set()
            for i in range(len(L_prev)):
                for j in range(i + 1, len(L_prev)):
                    a = L_prev[i]
                    b = L_prev[j]
        
                    prefix_a = a[:k-2]
                    prefix_b = b[:k-2]
        
                    if prefix_a == prefix_b and a[-1] != b[-1]:
                        new_candidate = tuple(sorted(set(a) | set (b)))
                        candidates.add(new_candidate)
            C_k = sorted(candidates)

        for combo in C_k:
            support = sum(1 for item in db if set(combo).issubset(item))
            if support >= minsup:
                L_k.append((combo, support))

        if not L_k:
            break

        else:
            frequent.extend(L_k)
            L_prev = [combo for combo, support in L_k]
            L_k = []
            k += 1

    frequent.sort(key=lambda x: (-x[1], -len(x[0]), x[0]))
    return frequent

In [8]:
fi_apriori = apriori(db, 2)
assert_equal(len(fi_apriori), 11)
assert_equal(fi_apriori[0], (('milk',), 5))

**Problem 6**

Create a function `apriori2` that accepts `db` and `minsup` then returns the maximal frequent itemsets as a list of tuples with their support using the Apriori algorithm. Sort the items in each tuple alphabetically. Sort the results, first by decreasing support, then by decreasing number of items then alphabetically. Use only the Python Standard Library.

In [9]:
def apriori2(db, minsup):
    """Find the maximal frequent itemsets in `db` using Apriori.

    An itemset is maximal if it is frequent and no other frequent
    itemset is a proper superset of it. This is computed by running
    `apriori` to get all frequent itemsets, then discarding any
    itemset that is a proper subset of another one in the result.

    Parameters
    ----------
    db : list of set
        The transaction database.
    minsup : int
        The minimum absolute support an itemset must have to be
        considered frequent.

    Returns
    -------
    list of tuple
        Each element is ``(itemset, support)`` for a maximal
        frequent itemset, with `itemset` a tuple of items sorted
        alphabetically. The list is sorted by decreasing support,
        then by decreasing itemset length, then alphabetically.
    """
    frequent = apriori(db, minsup)
    itemsets = [set(itemset) for itemset, _ in frequent]

    maximal = [
        (itemset, support)
        for i, (itemset, support) in enumerate(frequent)
        if not any(itemsets[i] < other for other in itemsets)
    ]

    maximal.sort(key=lambda x: (-x[1], -len(x[0]), x[0]))
    return maximal

In [10]:
maxfi_apriori = apriori2(db, 2)
assert_equal(len(maxfi_apriori), 3)
assert_equal(maxfi_apriori[0], (('eggs', 'milk', 'yogurt'), 2))

## FP-Growth Algorithm

The Apriori algorithm suffers from generating candidate itemsets that may not be in the database at all. FP-Growth, a pattern growth algorithm, avoids this problem by using projected databases.

FP-Growth, like any other pattern growth algorithm, uses an enumeration tree. This algorithm employs a particular kind of enumeration known as a prefix tree. Assume that there's a defined lexicographical order among items i.e., items can be sorted. For example, if we define the lexicographical order as alphabetical, then {milk, cheese} will be transformed to {cheese, milk} after sorting. A prefix tree is a tree where each node is an item and a higher-level node can only be connected to a lower-level node of later order (Figure 2).

<div style="width: 40em; margin: 0 auto">
    <img src="prefix-tree.png" style="width: 25em" />
    <p><strong>Figure 2</strong>. A higher-level prefix tree node can only be connected to a lower-level prefix tree node of later lexicographical order. Image source: Aggarwal (2015).</p>
</div>

What FP-growth does is it recursively scans the database to find the frequent items (not itemsets) but instead of scanning the entire database, it will only scan the subset of the database that contains the prefix of the itemsets and with the prefix removed from the values. This is known as the projected database.

To illustrate, FP-Growth will scan the database and finds that bread, with $minsup=2$, is a frequent itemset. The resulting projected database would then be:

| tid |           items      |
|-----|----------------------|
|  1  | {butter, milk}       |
|  3  | {cheese, eggs, milk} |

The algorithm would repeat the process but scanning the projected database instead. It would find that butter has only a support of 1 hence {bread, butter} is not a frequent itemset. It would then look at the other items (cheese, eggs, milk, in order) but it would find that only milk has a support passing the $minsup$ threshold. Thus, it would add {bread, milk} as a frequent itemset. However, the resulting projected database is empty so it won't go deeper. Instead it would go up, back to the original dataset where it would find that butter is not a frequent itemset but cheese is so it would repeat the same process for the projected database of cheese and so on.

The FP-growth algorithm is summarized below.

<img src="fp-growth.png" style="width: 40em" />

**Problem 7**

Create a function `fpgrowth` that accepts `db` and `minsup` then returns the frequent itemsets as a list of tuples with their support using the FP-Growth algorithm. Sort the items in each tuple alphabetically. Sort the results, first by decreasing support, then by decreasing number of items then alphabetically. Use only the Python Standard Library.

In [11]:
def fpgrowth(db, minsup):
    frequent = []
    
    sorted_db = [sorted(transaction) for transaction in db]
    stack = [(sorted_db, ())]
    
    while stack:
        current_db, prefix = stack.pop()
        
        item_counts = {}
        for transaction in current_db:
            for item in transaction:
                item_counts[item] = item_counts.get(item, 0) + 1
                
        frequent_items = []
        for item, count in item_counts.items():
            if count >= minsup:
                frequent_items.append(item)
        frequent_items.sort()
        
        for item in frequent_items:
            new_prefix = tuple(sorted(prefix + (item,)))
            frequent.append((new_prefix, item_counts[item]))
            
            projected_db = []
            for transaction in current_db:
                if item in transaction:
                    idx = transaction.index(item)
                    suffix = transaction[idx + 1:]
                    
                    if suffix:
                        projected_db.append(suffix)
                            
            if projected_db:
                stack.append((projected_db, new_prefix))

    frequent.sort(key=lambda x: (-x[1], -len(x[0]), x[0]))
    
    return frequent


In [12]:
def fpgrowth(db, minsup):
    """Find all frequent itemsets in `db` using an FP-growth style
    recursive projection.

    Instead of building an explicit FP-tree, this implementation
    keeps each (projected) transaction as a sorted list and grows
    itemsets depth-first: at each step it counts item frequencies in
    the current projected database, keeps the frequent items, and
    for each one recurses into the sub-database of transactions that
    contain it, restricted to the items that lexicographically
    follow it. This avoids ever generating a candidate itemset that
    does not already occur in the data.

    Parameters
    ----------
    db : list of set
        The transaction database.
    minsup : int
        The minimum absolute support an itemset must have to be
        considered frequent.

    Returns
    -------
    list of tuple
        Each element is ``(itemset, support)`` where `itemset` is a
        tuple of items sorted alphabetically. The list is sorted by
        decreasing support, then by decreasing itemset length, then
        alphabetically by itemset.
    """
    frequent = []
    sorted_db = [sorted(transaction) for transaction in db]
    stack = [(sorted_db, ())]

    while stack:
        current_db, prefix = stack.pop()

        item_counts = {}
        for transaction in current_db:
            for item in transaction:
                item_counts[item] = item_counts.get(item, 0) + 1

        frequent_items = sorted(
            item for item, count in item_counts.items() if count >= minsup
        )

        for item in frequent_items:
            new_prefix = tuple(sorted(prefix + (item,)))
            frequent.append((new_prefix, item_counts[item]))

            projected_db = []
            for transaction in current_db:
                if item in transaction:
                    idx = transaction.index(item)
                    suffix = transaction[idx + 1:]
                    if suffix:
                        projected_db.append(suffix)

            if projected_db:
                stack.append((projected_db, new_prefix))

    frequent.sort(key=lambda x: (-x[1], -len(x[0]), x[0]))
    return frequent


In [13]:
fi_fpgrowth = fpgrowth(db, 2)
assert_equal(len(fi_fpgrowth), 11)
assert_equal(fi_fpgrowth[0], (('milk',), 5))

# Association Pattern Mining

For many applications, frequent itemset analysis is not the final analysis performed but rather continues on to association pattern mining. Let us define a few terms first.

The **confidence** of an **association rule** $A \rightarrow B$ is the conditional probability that $B$ is in a transaction given that it contains $A$,
$$
conf(A \rightarrow B) = \Pr(B \in T_i | A \in T_i) = \frac{sup(A \cup B)}{sup(A)}.
$$
We say that $A$ is the **antecedent** and $B$ as the **consequent** of the rule. The objective of association pattern mining is to find the rules that have a confidence at least equal to a given minimum confidence $minconf$. **Lift** is the increase in the likelihood of $B$ in a transaction given that $A$ is already included,
$$
lift(A \rightarrow B) = \frac{conf(A \rightarrow B)}{relSup(B)}.
$$
A lift greater than 1 means $A$ and $B$ are more likely to be found together than just $B$ alone, a value of 1 means there is no association between $A$ and $B$, and a lift of less than 1 implies that $A$ and $B$ are unlikely to be together (negative association).

We find the association rules for a database of transactions by first looking for all frequent itemsets in the database with the minimum support. For all frequent itemsets, we generate the candidate association rules by setting one of the items as the consequent then all possible subsets of the remaining items as antecedent. The confidence of each candidate association rule is computed and only those that reached the minimum support is returned.

**Problem 8**

Create a function `assoc` that accepts the list of frequent itemsets returned by the functions above, minimum confidence $minconf$ and number of transactions in the database then returns the list of association rules with $minconf$ minimum confidence as a list of dicts. Each dict corresponds to a rule and it should have the following keys: `antecedent`, `consequent`, `support`, `confidence` and `lift`. Sort the rules by decreasing lift, decreasing confidence, decreasing support, consequent and antecedent. Use only the Python Standard Library.

In [14]:
def assoc(fi, minconf, db_size):
    """Generate association rules from a list of frequent itemsets.

    For every frequent itemset of size >= 2, each item in it is
    considered in turn as the (single-item) consequent, with the
    remaining items as the antecedent. A rule is kept if its
    confidence meets `minconf`.

    Parameters
    ----------
    fi : list of tuple
        Frequent itemsets as ``(itemset, support)`` pairs, e.g. the
        output of `brute`, `apriori`, or `fpgrowth`. Every subset of
        a kept itemset that is needed to compute confidence or lift
        must also appear in `fi`.
    minconf : float
        The minimum confidence a rule must have to be returned.
    db_size : int
        The number of transactions in the original database, used
        to convert absolute support into relative support for the
        lift calculation.

    Returns
    -------
    list of dict
        Each dict has keys `antecedent`, `consequent`, `support`,
        `confidence`, and `lift`. `consequent` is always a single
        item; `antecedent` is a single item if only one remains, or
        a tuple otherwise. The list is sorted by decreasing lift,
        decreasing confidence, decreasing support, then by
        consequent, then by antecedent.
    """
    support_lookup = {frozenset(itemset): support for itemset, support in fi}
    rules = []

    for itemset, support in fi:
        if len(itemset) < 2:
            continue

        for consequent in itemset:
            antecedent_items = tuple(
                item for item in itemset if item != consequent
            )
            antecedent_support = support_lookup.get(frozenset(antecedent_items))
            if antecedent_support is None:
                continue

            confidence = support / antecedent_support
            if confidence < minconf:
                continue

            consequent_support = support_lookup.get(frozenset((consequent,)))
            if consequent_support is None:
                continue

            relative_support_consequent = consequent_support / db_size
            lift = confidence / relative_support_consequent

            antecedent = (
                antecedent_items[0]
                if len(antecedent_items) == 1
                else antecedent_items
            )

            rules.append({
                "antecedent": antecedent,
                "consequent": consequent,
                "support": support,
                "confidence": confidence,
                "lift": lift,
            })

    def sort_key(rule):
        antecedent = rule["antecedent"]
        antecedent_key = (
            (antecedent,) if isinstance(antecedent, str) else antecedent
        )
        return (
            -rule["lift"],
            -rule["confidence"],
            -rule["support"],
            rule["consequent"],
            antecedent_key,
        )

    rules.sort(key=sort_key)
    return rules

In [15]:
rules = assoc(
    [
        (("a",), 5),
        (("a", "b"), 3),
        (("a", "c"), 3),
        (("b",), 3),
        (("c",), 3),
        (("a", "b", "c"), 2),
        (("a", "d"), 2),
        (("a", "e"), 2),
        (("b", "c"), 2),
        (("d",), 2),
        (("e",), 2),
    ],
    0.2,
    5,
)
assert_equal(len(rules), 13)
assert_equal(
    rules[0],
    {
        "antecedent": ("a", "c"),
        "consequent": "b",
        "support": 2,
        "confidence": 0.6666666666666666,
        "lift": 1.1111111111111112,
    },
)

# pyFIM

Pandas and scikit-learn do not have a frequent itemset analysis module. We could instead use [pyFIM](http://www.borgelt.net/pyfim.html), which is a rather comprehensive FIM library.

Here is an example of its usage:

In [16]:
import fim
fim.apriori(db)

ModuleNotFoundError: No module named 'fim'

Note that in `fim`, a positive argument to `supp` (support) is treated as a percentage and a negative value is considered as absolute support.

In [ ]:
fim.eclat(db, supp=-2)

The minimum length of an itemset can be specified with `zmin`.

In [ ]:
fim.fpgrowth(db, supp=20, zmin=2)

The `report` parameter is quite powerful and can be used, for example, to return the rules along with their lift as a fraction.

In [ ]:
fim.fpgrowth(db, target='r', supp=20, conf=30, report='l')

**Problem 9**

Compare the runtime of Apriori and FP-growth when the number of items is large, and when the number of transactions is large.

**When the number of items is large.** Apriori generates candidate $k$-itemsets combinatorially and re-scans the (projected or full) database once per level, so its candidate-generation step and its I/O both grow with the number of items even though the join-and-prune step discards candidates whose subsets aren't frequent. FP-growth never materializes an explicit candidate itemset that isn't already backed by data — it only recurses into items that actually co-occur — so it avoids most of the wasted candidate generation and tends to be substantially faster as the item universe grows, as long as the resulting FP-tree stays reasonably compact (small minsup relative to a very large, sparse item universe can still make the tree, and thus the recursion, large).

**When the number of transactions is large.** Apriori requires a full database scan for every level $k$, so its I/O cost scales with both $N$ and the number of levels — expensive when $N$ is large. FP-growth compresses the database into a (conditional) tree structure that shrinks at each recursive step, needing only a small, bounded number of full scans, so it typically handles large $N$ more gracefully than Apriori's repeated full passes.

**Overall.** FP-growth tends to win on both axes, which is why it largely superseded Apriori in practice, but it isn't free: building and holding the (conditional) trees in memory can become the bottleneck for very high-dimensional, sparse data, in which case Apriori's simpler, lower-memory candidate generation can be competitive despite the extra scans.

**Problem 10**

Suppose the owner of a store provided you their POS data (`Online Retail.xlsx`). Provide three suggestions to the owner of the store based on the results of FIM. More information about the dataset is available [here](https://archive.ics.uci.edu/ml/datasets/online+retail).

1. **Bundle frequently co-purchased items.** Novelty and seasonal gift items in this catalog tend to co-occur in the same basket (e.g. matching sets of tableware, decorations, or stationery). High-lift rules among these point to bundles or "frequently bought together" placements that raise average order value without discounting the anchor item.
2. **Use high-confidence rules to drive cross-sell recommendations and store layout.** A rule like $A \\rightarrow B$ with high confidence and lift $> 1$ says $B$ is a strong add-on once $A$ is in the basket, which the owner can act on directly, by prompting the recommendation at checkout for an online store, or by shelf placement for a physical one, rather than treating the catalog as unrelated SKUs.
3. **Flag negative or near-zero associations for pruning or re-merchandising.** Itemsets that never clear even a low `minsup`, or item pairs with lift close to or below 1, are not reinforcing each other's sales; that's a signal to reconsider promotion spend or shelf space on those SKUs rather than assuming every item benefits equally from foot traffic.

Any of these should be checked against confidence and lift, not support alone: a high-support rule can still have lift near 1, meaning the co-occurrence is just because both items are individually popular, not because they're actually associated.

# References

* C. Aggarwal, "Data Mining: The Textbook", Springer, 2015.
* P. Fournier‐Viger, J.C.W. Lin, B. Vo, T.T. Chi, J. Zhang, & H.B. Le, "A survey of itemset mining", Wiley Interdisciplinary Reviews: Data Mining and Knowledge Discovery, 7(4), e1207, 2017.
* U. Malik, "Association Rule Mining via Apriori Algorithm in Python", 2018 [retrieved from https://stackabuse.com/association-rule-mining-via-apriori-algorithm-in-python]